# Part C Storage Format Comparison
Use this notebook to collect evidence. Complete the marked answer cells in your own words.

In [1]:
from pathlib import Path
from time import perf_counter
from statistics import median
import pandas as pd

FILES = {
    'CSV': Path('02_retail_sales.csv'),
    'JSON': Path('03_retail_sales.json'),
    'Parquet': Path('04_retail_sales.parquet'),
}
for name, path in FILES.items():
    print(name, path.exists(), path.stat().st_size if path.exists() else 'missing')

CSV True 45506
JSON True 151870
Parquet True 23021


## Load the three equivalent datasets

In [2]:
loaders = {
    'CSV': lambda: pd.read_csv(FILES['CSV']),
    'JSON': lambda: pd.read_json(FILES['JSON'], lines=True),
    'Parquet': lambda: pd.read_parquet(FILES['Parquet']),
}
data = {name: loader() for name, loader in loaders.items()}
for name, frame in data.items():
    print(f'\n{name}: {frame.shape}')
    print(frame.dtypes)


CSV: (480, 15)
transaction_id      int64
order_id            int64
order_date         object
customer_id         int64
region             object
product_id          int64
product_name       object
category           object
quantity            int64
unit_price        float64
discount_rate     float64
payment_method     object
returned             bool
promotion_code     object
revenue           float64
dtype: object

JSON: (480, 15)
transaction_id      int64
order_id            int64
order_date         object
customer_id         int64
region             object
product_id          int64
product_name       object
category           object
quantity            int64
unit_price        float64
discount_rate     float64
payment_method     object
returned             bool
promotion_code     object
revenue           float64
dtype: object

Parquet: (480, 15)
transaction_id             int64
order_id                   int64
order_date        datetime64[ns]
customer_id                int64
region 

## Measure load time three times and report the median

In [3]:
timings = {}
for name, loader in loaders.items():
    runs = []
    for _ in range(3):
        start = perf_counter()
        _ = loader()
        runs.append((perf_counter() - start) * 1000)
    timings[name] = {'runs_ms': runs, 'median_ms': median(runs)}
timings

{'CSV': {'runs_ms': [3.7306660087779164,
   1.5813749632798135,
   1.266124949324876],
  'median_ms': 1.5813749632798135},
 'JSON': {'runs_ms': [5.405374977272004,
   3.232542017940432,
   3.2889999565668404],
  'median_ms': 3.2889999565668404},
 'Parquet': {'runs_ms': [2.1089999936521053,
   1.5739170485176146,
   1.332332962192595],
  'median_ms': 1.5739170485176146}}

In [4]:
comparison = pd.DataFrame([
    {
        'format': name,
        'file_size_bytes': FILES[name].stat().st_size,
        'median_load_ms': timings[name]['median_ms'],
        'rows': data[name].shape[0],
        'columns': data[name].shape[1],
    }
    for name in FILES
])
comparison

,format,file_size_bytes,median_load_ms,rows,columns
0,CSV,45506,1.581375,480,15
1,JSON,151870,3.289000,480,15
2,Parquet,23021,1.573917,480,15


## Student answers
Complete C1–C5 here. Add your comparison table, explanations, partitioning plan, and star-schema diagram.

## C1

CSV: structured. Every row has the same columns, set by one header row, like a table.
JSON: semi-structured. Each record has its own field names, so records can have different or missing fields.
Parquet: structured. It stores a fixed schema inside the file.

Parquet preserves types most explicitley. It is the only format where order_date loads as datetime64. CSV and JSON load it as object (text). The other columns (int64, float64, bool) load the same in all three formats, Parquet reads them from the stored schema.

## C2 

File sizes in bytes: 
CSV: 45506
JSON: 151870
Parquet: 23021


Load time: JSON was the slowest (3.29 ms), about twice as slow as CSV (1.58 ms) and Parquet (1.57 ms), because JSON repeats every field name in every record and must be parsed record by record.


| Format | File size (bytes) | Median load time (ms) | Rows | Columns | Data types |
|---|---|---|---|---|---|
| CSV | 45,506 | 1.58 | 480 | 15 | int64, float64, bool, object (`order_date` = object) |
| JSON | 151,870 | 3.29 | 480 | 15 | int64, float64, bool, object (`order_date` = object) |
| Parquet | 23,021 | 1.57 | 480 | 15 | int64, float64, bool, object (`order_date` = datetime64[ns]) |

## C3

**1. Exchanging data with a simple external system: CSV**

CSV is plain text that every progran can open, including Excel, databases, and simple scripts, so the other system needs no special software. It also has a small file size and a simple, flat layout.

**2. Keeping nested event-style records: JSON**

Each event can hold its own fields and can contain nested objects and lists, such as a bus log with a list of fault codes. Records can have different fields without breaking anything, and the field names make each record self-describing. CSV and Parquet need a fixed, flat structure.

**3. Repeated analytical queries over selected columns: Parquet**

Parquet stores data by column, so a query that needs only a few columns (such as `region` and `revenue`) reads just those columns and skips the others. It is also the smallest file and it keeps data types such as `order_date` as a real date, so the data is ready for analysis without conversion. The advantage is bigger with larger datasets.

## C4
**Partition key: order month**

Split the data into one folder per month, based on `order_date`:

Five years is 60 months, so there are 60 folders of about 8 million rows each.

**How partition pruning helps**

If you ask for March 2026 revenue, the system opens only the `month=2026-03` folder and ignores the others. The query is faster and cheaper.

**One poor choice: partition by `transaction_id`**

Every row has a different ID, so it would get up to 500 million folders with one row each. More time opening small files than reading the data, so queries become very slow.

## C5


```
                    +------------------+
                    |     DimDate      |
                    |------------------|
                    | date_key (PK)    |
                    | full_date        |
                    | year             |
                    | quarter          |
                    | month            |
                    | day_of_week      |
                    +--------+---------+
                             | 1
                             |
                             | *
+------------------+  +------+-----------------+  +------------------+
|   DimCustomer    |  |       FactSales        |  |    DimProduct    |
|------------------|  |------------------------|  |------------------|
| customer_key (PK)|1 | transaction_id (PK)    | 1| product_key (PK) |
| customer_id      +--+ date_key (FK)          +--+ product_id       |
| region           | *| customer_key (FK)      | *| product_name     |
+------------------+  | product_key (FK)       |  | category         |
                      | order_id               |  +------------------+
                      | quantity               |
                      | unit_price             |
                      | discount_rate          |
                      | revenue                |
                      | returned               |
                      | payment_method         |
                      | promotion_code         |
                      +------------------------+
```


### Dimensions

| Dimension | Primary key | Other columns | Foreign key in FactSales |
|---|---|---|---|
| `DimCustomer` | `customer_key` | `customer_id`, `region` | `customer_key` |
| `DimProduct` | `product_key` | `product_id`, `product_name`, `category` | `product_key` |
| `DimDate` | `date_key` | `full_date`, `year`, `quarter`, `month`, `day_of_week` | `date_key` |

### Denormalization
- **Benefit:** queries are simpler and faster. Category is stored directly in `DimProduct`, so revenue by category and month needs only joins from the fact table to two dimensions, with no chains of extra joins.
- **Cost:** repeated data and harder updates. A category name is stored on every product row that uses it, so renaming it means changing many rows, and a missed row creates inconsistent data.